# Qwen 4B / T4: kiểm thử pipeline trong 45 phút
Chọn **Runtime → Change runtime type → T4 GPU**. Dùng tài khoản Colab của bạn.
Notebook tải Qwen chính thức, nén NF4, chạy benchmark trên loopback và lưu Drive.
Không nhập API key. Model nhỏ không bảo đảm dự đoán được điểm Qwen3.7 Flash.
Tải lên `table_qa_colab_dev.zip` khi được hỏi; không upload `.env`.
Thời gian 45 phút bắt đầu sau xác thực Drive/nhập gói, trước cài thư viện và tải model.


In [ ]:
# Kết nối Drive và nhập gói phát triển (không cần khóa API)
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/TableQA_T4')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = DRIVE_ROOT / 'table_qa_colab_dev.zip'
if not BUNDLE.exists():
    uploaded = files.upload()
    if 'table_qa_colab_dev.zip' not in uploaded:
        raise ValueError('Hãy chọn table_qa_colab_dev.zip')
    BUNDLE.write_bytes(uploaded['table_qa_colab_dev.zip'])


In [ ]:
# Bắt đầu phiên 45 phút; mọi kết quả lưu trong một thư mục Drive mới
import hashlib, json, os, shutil, stat, subprocess, sys, time, zipfile
STARTED = time.time()
SESSION = DRIVE_ROOT / time.strftime('session_%Y%m%d_%H%M%S', time.gmtime(STARTED))
SESSION.mkdir(exist_ok=False)
WORK = Path('/content/table_qa_t4_' + str(int(STARTED)))
WORK.mkdir(exist_ok=False)
with zipfile.ZipFile(BUNDLE) as archive:
    if sum(i.file_size for i in archive.infolist()) > 100_000_000:
        raise ValueError('Gói vượt 100 MB')
    manifest = json.loads(archive.read('bundle_manifest.json'))
    for item in archive.infolist():
        target = (WORK / item.filename).resolve()
        if not target.is_relative_to(WORK.resolve()) or stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError('Đường dẫn ZIP không hợp lệ')
        if item.filename != 'bundle_manifest.json':
            if hashlib.sha256(archive.read(item)).hexdigest() != manifest['files'].get(item.filename):
                raise ValueError('Hash file không khớp')
    archive.extractall(WORK)
os.chdir(WORK)
# Để resume sau mất runtime: điền thư mục session cũ, không tự mở phiên mới.
RESUME_FROM = ''
if RESUME_FROM:
    for old in Path(RESUME_FROM).glob('*'):
        if old.is_dir() and (old / 'checkpoint.json').exists():
            shutil.copytree(old, SESSION / old.name)
    if (Path(RESUME_FROM) / 'selection.json').exists():
        shutil.copy2(Path(RESUME_FROM) / 'selection.json', SESSION / 'selection.json')
with (SESSION / 'bootstrap.log').open('w') as output:
    job = subprocess.Popen([sys.executable, '-m', 'benchmarks.colab_bootstrap',
        '--session-dir', str(SESSION), '--started', str(STARTED)], stdout=output, stderr=subprocess.STDOUT)
print('Session:', SESSION)


In [ ]:
# Theo dõi, dừng đúng hạn và giải phóng model khi kết thúc
try:
    while job.poll() is None and time.time() < STARTED + 2700:
        state_file = SESSION / 'job.json'
        state = json.loads(state_file.read_text()) if state_file.exists() else {'status': 'starting'}
        print(time.strftime('%H:%M:%S'), state['status'], flush=True)
        time.sleep(15)
finally:
    if job.poll() is None:
        job.terminate()
        try:
            job.wait(timeout=10)
        except subprocess.TimeoutExpired:
            job.kill()
            job.wait(timeout=5)
    # Bootstrap normally already stopped its children. Recover after an interruption.
    import psutil
    for process in psutil.Process().children(recursive=True):
        try:
            command = process.cmdline()
            if 'benchmarks.colab_server' in command and str(SESSION) in command:
                process.terminate()
        except psutil.NoSuchProcess:
            print('Model process already exited')
    from benchmarks.colab_io import export_results
    print('Kết quả:', export_results(SESSION))
    if (SESSION / 'summary.json').exists():
        print((SESSION / 'summary.json').read_text()[:12000])


Kết quả local nằm trong `summary.json`, CSV và checkpoint từng stage.
Chỉ so sánh các stage có `complete=true`. Holdout chạy sau khi khóa `selection.json`.
Nếu hết giờ/quota hoặc setup lỗi, xem `job.json`, `install.log`, `server.log`.
Kết quả này không phải benchmark model chấm thi. Không đổi mặc định `run.py`.
Khi xong, **Runtime → Disconnect and delete runtime** để giải phóng tài nguyên.
